In [1]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/pre_cell_10.pickle

In [3]:
%%RecordEvent
%%time
### cell 10 ###

# Compute appropriate LOS difference per player per play in one pass
los_diff = (
    df.groupby(['gameId', 'playId', 'nflId'], as_index=False)
      .agg(
          max_diff=('los_diff', 'max'),
          min_diff=('los_diff', 'min'),
          posTeam=('posTeam', 'first')
      )
      .assign(
          los_diff=lambda d: d['max_diff'].where(d['posTeam'] == 1, d['min_diff'])
      )
      .groupby('nflId', as_index=False)['los_diff']
      .mean()
)
# Merge back onto snap_speed and rename columns
snap_speed = (
    snap_speed
      .merge(los_diff, on='nflId')
      .rename(columns={
          's': 'snap_s',
          'a': 'snap_a',
          'los_diff': 'snap_los_diff'
      })
)

CPU times: user 25.7 ms, sys: 4.28 ms, total: 30 ms
Wall time: 30 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/nickwan/creating-player-stats-using-tracking-data/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_10_try_1.pickle